In [ ]:
import os
import time
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

def set_seed(seed) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FERImageDataset(Dataset):
    def __init__(self, image_dir, csv_file, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.file_paths = []
        self.labels = []

        df = pd.read_csv(csv_file)

        for _, row in df.iterrows():
            file_name = str(row['image_name']).strip()
            label = int(row['label'])

            if file_name.endswith('.jpg'):
                img_path = os.path.join(image_dir, file_name)

                if os.path.exists(img_path):
                    self.file_paths.append(img_path)
                    self.labels.append(label)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        image = Image.open(img_path).convert('L')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

class CNNImageClassifier(nn.Module):
    def __init__(self, num_classes=7, dropout=0.5):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.conv_block(x)
        return self.classifier(features)

def evaluate(model: nn.Module, loader: DataLoader, class_weights=None) -> dict:
    model.eval()
    all_y = []
    all_pred = []
    total_loss = 0.0
    n = 0

    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = loss_fn(logits, y)

            pred = logits.argmax(dim=1)
            all_y.append(y.cpu().numpy())
            all_pred.append(pred.cpu().numpy())
            total_loss += loss.item() * y.size(0)
            n += y.size(0)

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    return {
        "loss": total_loss / max(1, n),
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    class_weights=None,
    lr: float = 1e-3,
    max_epochs: int = 20,
    weight_decay: float = 0.0,
    clip_grad_norm: float = None,
    patience: int = 3,
) -> list:

    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    optim = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_val_f1 = 0.0
    bad_epochs = 0

    hist = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        t0 = time.perf_counter()

        total_loss = 0.0
        n = 0
        correct = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optim.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()

            if clip_grad_norm is not None:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)

            optim.step()

            total_loss += loss.item() * y.size(0)
            n += y.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()

        train_loss = total_loss / max(1, n)
        train_acc = correct / max(1, n)
        val = evaluate(model, val_loader, class_weights)
        dt = time.perf_counter() - t0

        record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val["loss"],
            "val_acc": val["acc"],
            "val_f1": val["f1"],
            "time_s": dt,
        }

        hist.append(record)

        print(
            f"epoch {epoch:02d} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val['loss']:.4f} acc {val['acc']:.4f} f1 {val['f1']:.4f} | "
            f"time {dt:.1f}s"
        )

        if patience is not None:
            if val["f1"] > best_val_f1 + 1e-4:
                best_val_f1 = val["f1"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print("Early stopping triggered!")
                    if best_state is not None:
                        model.load_state_dict(best_state)
                    break

    if patience is not None and best_state is not None:
        model.load_state_dict(best_state)

    return hist

In [2]:
!mkdir -p /content/local_images

!cp -rL /content/drive/MyDrive/Stratified_10k_Cleaned_64x64/. /content/local_images/

mkdir: /content: Read-only file system
cp: /content/local_images: No such file or directory


In [ ]:
dataset = FERImageDataset(
    image_dir='/content/local_images',
    csv_file='/content/drive/MyDrive/Copy of Stratified_10k_Metadata.csv',
    transform=transform
)

labels = dataset.labels
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float).to(device)
print(f"Computed Class Weights: {class_weights_np}")

indices = list(range(len(dataset)))
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=7
)

train_subset = Subset(dataset, train_idx)
val_subset = Subset(dataset, val_idx)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

model = CNNImageClassifier(num_classes=7, dropout=0.5).to(device)

history = fit(
    model,
    train_loader,
    val_loader,
    class_weights=class_weights_tensor,
    lr=1e-3,
    max_epochs=30,
    patience=5
)

Computed Class Weights: [0.95751603 0.95495753 1.39198776 0.95304762 0.95495753 0.95687512
 0.95368341]
Initializing model and starting training...
epoch 01 | train loss 1.9429 acc 0.1595 | val loss 1.9225 acc 0.1873 f1 0.1404 | time 111.6s
epoch 02 | train loss 1.8743 acc 0.2357 | val loss 1.7854 acc 0.2697 f1 0.2423 | time 31.1s
epoch 03 | train loss 1.7200 acc 0.3235 | val loss 1.6668 acc 0.3387 f1 0.2872 | time 30.7s
epoch 04 | train loss 1.6239 acc 0.3679 | val loss 1.5992 acc 0.3806 f1 0.3384 | time 30.8s
epoch 05 | train loss 1.5503 acc 0.4106 | val loss 1.5414 acc 0.3951 f1 0.3732 | time 31.1s
epoch 06 | train loss 1.4667 acc 0.4377 | val loss 1.5251 acc 0.4126 f1 0.3968 | time 30.9s
epoch 07 | train loss 1.4052 acc 0.4725 | val loss 1.5078 acc 0.4176 f1 0.4126 | time 31.4s
epoch 08 | train loss 1.3422 acc 0.4927 | val loss 1.5024 acc 0.4211 f1 0.4165 | time 31.3s
epoch 09 | train loss 1.2779 acc 0.5234 | val loss 1.4898 acc 0.4386 f1 0.4284 | time 31.2s
epoch 10 | train loss 1

In [1]:
import torch

save_path = '/content/drive/MyDrive/FER_baseline_model_weights_2.pth'

torch.save(model.state_dict(), save_path)

NameError: name 'model' is not defined